In [13]:
import sys
sys.path.append('..')

In [14]:
from rfdetr import RFDETRSmall

model = RFDETRSmall()

detections = model.predict("../media/frames/first-half/cropped.jpg", threshold=0.4)

[2026-10-13 20:54:42] [INFO] rf-detr - File /Users/luke/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


[2026-10-13 20:54:42] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-13 20:54:42] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-10-13 20:54:43] [INFO] rf-detr - File /Users/luke/.roboflow/models/rf-detr-small.pth already exists with correct MD5 hash.


[2026-10-13 20:54:45] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).


In [15]:
import supervision as sv
from rfdetr.assets.coco_classes import COCO_CLASSES

labels = [f"{COCO_CLASSES[class_id]}" for class_id in detections.class_id]

annotated_image = sv.BoxAnnotator().annotate(detections.metadata["source_image"], detections)
annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

In [16]:
from src.video.extract_frames import extract_frames

frames = extract_frames('../media/videos/test.mp4', 0.5)

In [20]:
import cv2
import supervision as sv
from rfdetr import RFDETRNano
from PIL import Image

model = RFDETRNano()
tracker = sv.ByteTrack(frame_rate=2, lost_track_buffer=5)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_frames = []

for frame in frames:
    # Convert OpenCV BGR to RGB
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # RF-DETR returns supervision.Detections
    detections = model.predict(
        Image.fromarray(rgb),
        threshold=0.4
    )

    # Detect and track players only
    players = detections[detections.class_id == 1]
    tracked = tracker.update_with_detections(players)

    labels = [
        f"#{tracker_id}"
        for tracker_id in tracked.tracker_id
    ]

    annotated = box_annotator.annotate(
        scene=frame.copy(),
        detections=tracked
    )

    annotated = label_annotator.annotate(
        scene=annotated,
        detections=tracked,
        labels=labels
    )

    annotated_frames.append(annotated)

[2026-10-13 20:56:01] [INFO] rf-detr - File /Users/luke/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


[2026-10-13 20:56:01] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-10-13 20:56:01] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-10-13 20:56:04] [INFO] rf-detr - File /Users/luke/.roboflow/models/rf-detr-nano.pth already exists with correct MD5 hash.


In [18]:
frame = next(frames)

rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
detections = model.predict(Image.fromarray(rgb), threshold=0.4)

tracker = sv.ByteTrack(frame_rate=2, lost_track_buffer=5)

players = detections[detections.class_id == 1]
tracked = tracker.update_with_detections(players)

print("Players detected:", len(players))
print("Players tracked:", len(tracked))
print("Tracking IDs:", tracked.tracker_id)

labels = [
    f"#{tracker_id}"
    for tracker_id in tracked.tracker_id
]

# Draw ALL detected players, even if ByteTrack rejects them
annotated = box_annotator.annotate(
    scene=frame.copy(),
    detections=players
)

annotated = label_annotator.annotate(
        scene=annotated,
        detections=tracked,
        labels=labels
    )

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

StopIteration: 

In [21]:
import matplotlib.pyplot as plt
import cv2

print("Number of frames:", len(annotated_frames))
print("Frame shape:", annotated_frames[0].shape)

plt.figure(figsize=(16, 9))
plt.imshow(cv2.cvtColor(annotated_frames[4], cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

Number of frames: 0


IndexError: list index out of range